# Entrega de Proyecto Final: IA Generativa
## Asistente Experto con Llama 3.3 (Groq), RAG y Agentes

**Alumno (UUID):** 57fdfdd2-0056-4bdb-8c56-da5a86b58304

**Dominio Elegido:** Visualización de Datos y Comunicación Efectiva para Presentaciones.

Este notebook contiene el MVP requerido para la evaluación, integrando:
- Creación de Base de Conocimiento con ChromaDB y Embeddings locales (HuggingFace / Sentence-Transformers).
- Uso del LLM Llama 3.3 70B a través de Groq para tareas de Retrieval-Augmented Generation (RAG).
- Construcción del flujo conversacional con memoria usando LangGraph.

## 1. Instalación de Dependencias y Configuración
Asegúrate de tener instaladas las dependencias listadas en `requirements.txt` y tu API Key de Groq en un archivo `.env`.


In [1]:
%pip install -qU langchain langgraph langchain-openai langchain-huggingface sentence-transformers chromadb python-dotenv pypdf


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import time
import shutil
from dotenv import load_dotenv

# Cargar variables de entorno (archivo .env con GROQ_API_KEY)
load_dotenv()

if "GROQ_API_KEY" not in os.environ:
    print("API KEY no encontrada. Asegúrate de tener un archivo .env con GROQ_API_KEY=tu_clave")
else:
    print("API Key de Groq cargada correctamente.")


API Key de Groq cargada correctamente.


## 2. Cargar Documentos y Crear Base de Conocimiento (ChromaDB)
Vamos a cargar los documentos de texto sobre visualización de datos y guardarlos en una base de datos vectorial.


In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Cargar documentos
loader_md = DirectoryLoader('./data', glob="**/*.md", loader_cls=TextLoader)
loader_pdf = DirectoryLoader('./data', glob="**/*.pdf", loader_cls=PyPDFLoader)
docs = loader_md.load() + loader_pdf.load()
print(f"Documentos cargados: {len(docs)}")

# 2. Dividir documentos en fragmentos (chunks)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
print(f"Fragmentos creados: {len(splits)}")

# 3. Crear embeddings y almacenar en ChromaDB
# Usamos Sentence-Transformers con un modelo multilingüe optimizado para español
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Limpiar base vectorial anterior si existe (necesario al cambiar modelo de embeddings)
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")

vectorstore = Chroma.from_documents(
    documents=splits, embedding=embeddings, persist_directory="./chroma_db"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Base de conocimiento creada exitosamente.")


C:\Users\xluna\AppData\Local\Temp\ipykernel_25128\2936534302.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader
c:\Users\xluna\Desktop\Proyecto\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Documentos cargados: 2
Fragmentos creados: 16


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6817.72it/s]


Base de conocimiento creada exitosamente.


## 3. Configuración del LLM y Herramienta RAG
Creamos el retriever tool y configuramos Llama 3.3 70B a través de Groq como LLM principal. Se utiliza la API de Groq, que es compatible con el protocolo OpenAI.


In [4]:
from langchain_core.tools import create_retriever_tool
from langchain_openai import ChatOpenAI

retriever_tool = create_retriever_tool(
    retriever,
    "busqueda_visualizacion_datos",
    "Busca y devuelve información sobre visualización de datos, storytelling, diseño de presentaciones y comunicación efectiva."
)

tools = [retriever_tool]

# Inicializamos Llama 3.3 70B a través de Groq
llm = ChatOpenAI(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    temperature=0.3
)
llm_with_tools = llm.bind_tools(tools)

print("LLM (Llama 3.3 70B via Groq) configurado correctamente.")


LLM (Llama 3.3 70B via Groq) configurado correctamente.


## 4. Definición del System Prompt
Configuramos el rol del agente experto. Este prompt define su personalidad, conocimiento y cómo debe interactuar.


In [5]:
from langchain_core.messages import SystemMessage

# System Prompt justificado:
# - Se define el rol (experto) para establecer autoridad.
# - Se instruye a usar la herramienta de búsqueda para ser preciso y evitar alucinaciones.
# - Se le pide claridad, el uso de la 'regla del 3' y un tono educativo.
# - Se incluyen instrucciones sobre cómo actuar si no sabe algo.
system_prompt = """Eres un experto consultor en Visualización de Datos y Comunicación Efectiva.
Tu objetivo es ayudar a profesionales a preparar charlas importantes sobre datos, mejorar sus gráficos y construir narrativas sólidas.

Sigue estas reglas estrictamente:
1. Usa la herramienta 'busqueda_visualizacion_datos' para buscar en tu base de conocimientos siempre que te pregunten sobre principios de diseño, storytelling o presentaciones.
2. Si la información no está en tu base de conocimiento, admítelo y sugiere buenas prácticas generales.
3. Responde de forma clara, estructurada y educativa. Usa la regla del 3 (agrupar ideas en tres puntos).
4. Fomenta el diseño siguiendo el principio KISS.
5. Mantén un tono alentador y profesional.
"""



## 5. Construcción del Grafo (LangGraph) con Memoria
Creamos el grafo para manejar el estado de la conversación (memoria) y el flujo entre el LLM y la herramienta de búsqueda.


In [6]:
from typing import Annotated, Sequence, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# Definimos el estado
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

# Nodo del agente
def call_model(state: AgentState):
    messages = state["messages"]
    # Inyectar el system prompt al inicio
    if not isinstance(messages[0], SystemMessage):
        messages = [SystemMessage(content=system_prompt)] + messages
    
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# Construir el grafo
workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)

# Nodo de herramientas preconstruido de LangGraph
tool_node = ToolNode(tools)
workflow.add_node("tools", tool_node)

# Conexiones
workflow.add_edge(START, "agent")
workflow.add_conditional_edges(
    "agent",
    tools_condition,
)
workflow.add_edge("tools", "agent")

# Añadimos memoria (checkpointer)
memory = MemorySaver()

# Compilar grafo
app = workflow.compile(checkpointer=memory)

print("Agente compilado exitosamente.")


Agente compilado exitosamente.


## 6. Interacción: Ejemplos de Conversación (Prueba de Memoria)
Tal como se solicita en los requisitos, a continuación se presentan 5 preguntas de ejemplo interactuando con el agente. 
El **Ejemplo 3** pone a prueba la memoria del agente, ya que hace referencia al turno anterior sin especificar el contexto de nuevo.

In [7]:
# Configuración para usar la misma sesión/hilo de memoria
config = {"configurable": {"thread_id": "sesion_1"}}

def chat_con_agente(pregunta, max_reintentos=3):
    """Envía una pregunta al agente con reintentos automáticos ante errores de la API."""
    print(f"\nUsuario: {pregunta}")
    inputs = {"messages": [HumanMessage(content=pregunta)]}
    
    for intento in range(1, max_reintentos + 1):
        try:
            for output in app.stream(inputs, config=config, stream_mode="values"):
                last_message = output["messages"][-1]
            
            # Imprimir la respuesta final del agente
            print(f"Agente: {last_message.content}")
            return  # Éxito
            
        except Exception as e:
            error_str = str(e).lower()
            if "429" in str(e) or "rate" in error_str or "limit" in error_str:
                espera = 15 * intento
                print(f"Rate limit alcanzado (intento {intento}/{max_reintentos}). "
                      f"Esperando {espera}s...")
                time.sleep(espera)
            else:
                print(f"Error: {e}")
                return
    
    print(f"Se agotaron los {max_reintentos} reintentos.")

# Pausa entre ejemplos para respetar rate limits
PAUSA = 5  # segundos

# Ejemplo 1: Pregunta general (usará la base de conocimiento)
chat_con_agente("¿Qué es el 'chartjunk' y por qué debería evitarlo?")
time.sleep(PAUSA)

# Ejemplo 2: Pregunta específica sobre gráficos
chat_con_agente("Quiero comparar las ventas de 10 productos con nombres muy largos. ¿Qué gráfico uso?")
time.sleep(PAUSA)

# Ejemplo 3: Poniendo a prueba la memoria (referencia al turno anterior)
chat_con_agente("¿Y qué color me recomiendas usar para resaltar el producto con más ventas en ese gráfico?")
time.sleep(PAUSA)

# Ejemplo 4: Conceptos de presentación
chat_con_agente("Tengo una diapositiva con un gráfico súper importante, ¿qué texto debo ponerle al lado?")
time.sleep(PAUSA)

# Ejemplo 5: Storytelling
chat_con_agente("¿Cómo estructuro la historia para mi presentación usando la narrativa clásica?")

print("\nTodos los ejemplos ejecutados.")



Usuario: ¿Qué es el 'chartjunk' y por qué debería evitarlo?
Agente: El 'chartjunk' se refiere a los elementos innecesarios o distractores en una visualización de datos, como gráficos 3D, colores brillantes, fondos complejos, leyendas redundantes, etc. Estos elementos pueden dificultar la comprensión de la información y restar claridad a la presentación.

Para evitar el 'chartjunk', es importante seguir los principios de diseño clásicos y mantener la simplicidad en las visualizaciones de datos. Algunos consejos para evitar el 'chartjunk' incluyen:

1. **Mantener la simplicidad**: Utilizar gráficos y elementos visuales simples y fáciles de entender.
2. **Eliminar elementos innecesarios**: Quitar cualquier elemento que no aporte valor a la visualización de datos.
3. **Utilizar colores y fondos adecuados**: Utilizar colores y fondos que no distraigan y que permitan una fácil lectura de la información.
4. **Optimizar la legibilidad**: Asegurarse de que la información sea fácil de leer y en

## 7. Celda Interactiva 
Ejecuta esta celda para probar conversar directamente con el agente. Escribe 'salir' para terminar el bucle de interacción.

In [16]:
print("Hola! Soy tu asistente en Visualización de Datos. Escribe 'salir' para terminar.")
config_interactivo = {"configurable": {"thread_id": "sesion_interactiva"}}

while True:
    user_input = input("Tu: ")
    if user_input.lower() in ['salir', 'exit', 'quit']:
        print("Hasta luego!")
        break
    
    inputs = {"messages": [HumanMessage(content=user_input)]}
    try:
        for output in app.stream(inputs, config=config_interactivo, stream_mode="values"):
            last_message = output["messages"][-1]
        
        print(f"\nAgente:\n{last_message.content}\n")
    except Exception as e:
        print(f"\nError: {e}\n")
    print("-" * 50)


Hola! Soy tu asistente en Visualización de Datos. Escribe 'salir' para terminar.
Hasta luego!
